# 00 — Challenge solution

Debrief only. Run this in the same kernel as the student notebook, after the setup cell, so `client`, `model`, `price_in`, and `price_out` already exist.

This is the last streaming cell, pointed at a joke: print, append, keep `usage`, then turn tokens into dollars.


In [11]:
from pathlib import Path
import os

from dotenv import load_dotenv
from openai import OpenAI


def find_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Start Jupyter from the course directory, "
        "or copy .env.example to .env next to pyproject.toml."
    )


ROOT = find_root()
load_dotenv(ROOT / ".env")

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
price_in = float(os.environ["PRICE_INPUT_PER_MILLION"])
price_out = float(os.environ["PRICE_OUTPUT_PER_MILLION"])

assert api_key, "OPENAI_API_KEY is missing. Copy .env.example to .env and add the class key."
assert model, "MODEL_DEFAULT is missing from .env."

print("repo root:", ROOT)
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print(f"prices: ${price_in}/1M input, ${price_out}/1M output")
client = OpenAI()

repo root: /Users/tarekatwan/Downloads/ai_agents_course
OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
prices: $0.2/1M input, $1.25/1M output


In [12]:
text = ""
usage = None

stream = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Tell a one-sentence joke about invoices."}],
    max_completion_tokens=128,
    reasoning_effort="none",
    stream=True,
    stream_options={"include_usage": True},
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")
        text = text + chunk.choices[0].delta.content
    if chunk.usage:
        usage = chunk.usage

print()

prompt_tokens = usage.prompt_tokens
cost = prompt_tokens / 1_000_000 * price_in + usage.completion_tokens / 1_000_000 * price_out
print(f"prompt_tokens={prompt_tokens}  cost=${cost:.6f}")

I tried to tell an invoice a joke, but it just kept asking for “payment terms”—turns out it’s a real stickler for punchlines.


prompt_tokens=15  cost=$0.000047


In [13]:
assert text and text.strip(), "text should be the assembled joke"
assert prompt_tokens > 0, "prompt_tokens should come from usage"
assert cost > 0, "cost should be dollars, greater than zero"
print("looks good")

looks good
